**Import packages and dependecies**

In [1]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


ERROR: file:///C:/Users/tcphan/OneDrive/Documents/Data%20Science%20Projects/topological does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import kagglehub

from tda.rips import VietorisRips
from tda.distance import bottleneck_distance, wasserstein_distance
from tda.plotting import plot_persistence_diagram

### 1. Load in data inputs

The dataset shown below is from the Digital Payment Fraud Detection Benchmark which contains simulated, large-scale digital payment transaction data. The data spans one full calendar year with a known feature drift occurring mid-year. We will use this data to see how TDA can be applied to detect this feature drift

**Months 1-6**
- Merchant-driven fraud influence
- Risk primarily influenced by merchant and IP-level signals

**Months 7-12**
- Increased velocity-based fraud
- Low-amount micro-transactions
- Higher sensitivity to international activity

For further information about the data, see [Digital Payment Fraud Detection Benchmark](https://www.kaggle.com/datasets/rohit8527kmr7518/digital-payment-fraud-detection-benchmark?select=transactions_train.csv).

In [3]:
# Download data fram Kaggle
path = kagglehub.dataset_download(
    "rohit8527kmr7518/digital-payment-fraud-detection-benchmark"
)

# Train contains all transactions for months 1-6
transactions_train_df = pl.read_csv(f"{path}/transactions_train.csv")
# Test contains all transactions for months 7-12
transactions_test_df = pl.read_csv(f"{path}/transactions_test.csv")
# Combine data
all_transactions_df = pl.concat([transactions_train_df, transactions_test_df])

# Create month column
all_transactions_df = all_transactions_df.with_columns(
    pl.col("transaction_time").str.to_datetime().alias("transaction_time")
)
all_transactions_df = all_transactions_df.with_columns(
    pl.col("transaction_time").dt.month().alias("month_num"),
    pl.col("transaction_time").dt.to_string("%b").alias("month_short_name"),
)

# Show transactions
all_transactions_df.limit(50).show()

transaction_id,transaction_time,customer_id,merchant_id,account_age_days,credit_score_band,kyc_level,avg_monthly_spend,merchant_risk_score,transaction_amount,payment_channel,device_type,is_international,ip_risk_score,txn_count_1h,txn_count_24h,failed_txn_count_24h,geo_distance_from_last_txn,amount_deviation_from_user_mean,is_fraud,post_auth_risk_score,month_num,month_short_name
i64,datetime[μs],i64,i64,i64,i64,i64,f64,f64,f64,str,str,i64,f64,i64,i64,i64,f64,f64,i64,f64,i8,str
359131,2023-01-01 00:02:00.328105,11102,2282,284,2,3,6091.747132,0.456269,2408.320473,"""wallet""","""desktop""",0,0.142532,1,3,1,33.458018,2205.262235,0,0.09992,1,"""Jan"""
351207,2023-01-01 00:02:26.339769,22891,3016,1363,2,3,3794.044563,0.449021,2765.255095,"""bank_transfer""","""mobile""",0,0.131811,0,5,0,3.375083,2638.786943,0,0.291715,1,"""Jan"""
10209,2023-01-01 00:06:54.145825,3102,1855,1318,5,2,6697.058451,0.220252,1529.079168,"""card""","""desktop""",0,0.322137,0,5,0,13.732603,1305.843886,0,0.216647,1,"""Jan"""
62660,2023-01-01 00:06:57.723185,4041,2525,1914,1,1,2906.711704,0.202223,610.407487,"""card""","""mobile""",0,0.171764,1,2,0,18.840187,513.517097,0,0.354154,1,"""Jan"""
384254,2023-01-01 00:08:05.487541,3979,1555,360,2,3,5082.651983,0.17123,986.397163,"""card""","""mobile""",0,0.248766,1,1,0,15.344375,816.97543,0,0.149084,1,"""Jan"""


List of primary keys, target outcomes, and features fields.

In [4]:
primary_keys_list = [
    "transaction_id",
    "transaction_time",
    "month_num",
    "month_short_name",
    "customer_id",
    "merchant_id",
]

target_outcomes_list = [
    "is_fraud",
    "post_auth_risk_score",
]

features_list = [
    "account_age_days",
    "credit_score_band",
    "kyc_level",
    "avg_monthly_spend",
    "merchant_risk_score",
    "transaction_amount",
    "payment_channel",
    "device_type",
    "is_international",
    "ip_risk_score",
    "txn_count_1h",
    "txn_count_24h",
    "failed_txn_count_24h",
    "geo_distance_from_last_txn",
    "amount_deviation_from_user_mean",
]

print(f"Total  # of primary keys: {len(primary_keys_list)}")
print(f"Total # of target outcomes: {len(target_outcomes_list)}")
print(f"Total # of features: {len(features_list)}")

Total  # of primary keys: 6
Total # of target outcomes: 2
Total # of features: 15


### 2. Apply data preprocessing

Remove columns if they have high missingness rate.

In [5]:
# Initialize parameters
tot_n_rows = all_transactions_df.shape[0]
missing_threshold = (
    0.4  # Columns w/ missing rate less than or equal to threshold are kept
)

# Calculate the percent of missing in each column
missing_count_df = all_transactions_df.null_count() / tot_n_rows
missing_count_df = missing_count_df.unpivot(
    on=features_list, variable_name="Variable Name", value_name="p_missing"
)

# Remove columns from data
columns_failed_threshold_list = (
    missing_count_df.filter(pl.col("p_missing") > missing_threshold)
    .select("Variable Name")
    .to_series()
)
preprocessed_transactions_df = all_transactions_df.drop(*columns_failed_threshold_list)
features_list = [c for c in features_list if c not in columns_failed_threshold_list]
print(
    f"Total # of columns removed due to high missing rate: {len(columns_failed_threshold_list)}"
)

Total # of columns removed due to high missing rate: 0


Apply label encoding to all string data type columns.

In [6]:
# List of all string type columns
string_dtype_list = [
    name
    for name, dtype in preprocessed_transactions_df.select(features_list).schema.items()
    if dtype == pl.String
]

# Apply label encoding to convert string to integer format
for colname in string_dtype_list:
    preprocessed_transactions_df = preprocessed_transactions_df.with_columns(
        pl.col(colname).cast(pl.Categorical).to_physical().alias(f"{colname}")
    )

print(f"Total # of string features requiring label encoding: {len(string_dtype_list)}")
preprocessed_transactions_df.select(string_dtype_list).show()

Total # of string features requiring label encoding: 2


payment_channel,device_type
u32,u32
3,0
2,1
0,0
0,1
0,1


Count the number of records per month.

In [7]:
n_records_per_month = (
    preprocessed_transactions_df.group_by(["month_num", "month_short_name"])
    .agg(pl.len().alias("n_records"))
    .sort("month_num")
)
n_records_per_month.show(12)

month_num,month_short_name,n_records
i8,str,u32
1,"""Jan""",33881
2,"""Feb""",31006
3,"""Mar""",34288
4,"""Apr""",32827
5,"""May""",34042
6,"""Jun""",32950
7,"""Jul""",34023
8,"""Aug""",34207
9,"""Sep""",32889


### 3. Calculate distances between persistence diagrams over time

Calculate the Rips simplices for each month. Due to the large size of the data, calculating the Rips simplices for all observations can take a long time. As such, to reduce computation time, we take a random subsample of the data to evaluate the Rips simplices.

In [10]:
# Number of dimensions to calculate simplices for
n_dimensions = 2
# The maximum distance between each data point to consider
max_epsilon = 50.0

# Number of starting months to represent the 'base' that subsequent months will be compared against
base_months = 3
# Percentage of observations to randomly sample
sample_fraction = 0.25
# Number of months
tot_n_months = preprocessed_transactions_df.select(
    "month_num"
).n_unique()


# Calculate Rips for the base months
base_df = (
    preprocessed_transactions_df.filter(pl.col("month_num") <= base_months)
    .select(features_list)
    .sample(fraction=sample_fraction, seed=42)
)
base_rips_config = VietorisRips(max_dim=n_dimensions, max_epsilon=max_epsilon)
base_rips_config.fit_transform(base_df)

print(f"\nBase Months <= {base_months}")
print("-" * 100)

for i in range(n_dimensions + 1):
    print(
        f"# of simplices for {i}-th dimension: {len(base_rips_config.get_simplices_for_dim(dim=i)):,}"
    )


# Calculate the Rips for each subsequent comparison months
comp_rips_config_dict = {}
for m in range(base_months+1, tot_n_months+1):

    # Limit to features only and sample a subset of the data in the given month
    df = (
        preprocessed_transactions_df.filter(pl.col("month_num") == m)
        .select(features_list)
        .sample(fraction=sample_fraction, seed=42)
    )

    # Calculate Rips simplices
    rips_config = VietorisRips(max_dim=n_dimensions, max_epsilon=max_epsilon)
    rips_config.fit_transform(df)

    print(f"\nMonth {m}")
    print("-" * 100)

    for i in range(n_dimensions + 1):
        print(
            f"# of simplices for {i}-th dimension: {len(rips_config.get_simplices_for_dim(dim=i)):,}"
        )

    # Update dictionary
    comp_rips_config_dict[m] = rips_config



Base Months <= 3
----------------------------------------------------------------------------------------------------
# of simplices for 0-th dimension: 24,793
# of simplices for 1-th dimension: 803
# of simplices for 2-th dimension: 13

Month 4
----------------------------------------------------------------------------------------------------
# of simplices for 0-th dimension: 8,206
# of simplices for 1-th dimension: 82
# of simplices for 2-th dimension: 1

Month 5
----------------------------------------------------------------------------------------------------
# of simplices for 0-th dimension: 8,510
# of simplices for 1-th dimension: 94
# of simplices for 2-th dimension: 2

Month 6
----------------------------------------------------------------------------------------------------
# of simplices for 0-th dimension: 8,237
# of simplices for 1-th dimension: 87
# of simplices for 2-th dimension: 1

Month 7
---------------------------------------------------------------------------

Calculate the Wasserstein distance to determine how far apart the the persistence diagram between the current and prior months are from one another. We see that, as expected, there is a signifant drop startin

In [ ]:
# Get the birth-death pairs for the base months
base_birth_death_by_dimension = base_rips_config.compute_birth_death_pairs()
base_birth_death_pairs_list = [
    pair
    for pairs_list in base_birth_death_by_dimension.values()
    for pair in pairs_list
    if pair[1] != np.inf
]

# Get the birth-death pairs for each comparison month
distances_dict = {}
for month_num, rips_config in comp_rips_config_dict.items():

    # Calculate birth-death pairs
    comp_birth_death_by_dimension = rips_config.compute_birth_death_pairs()

    # Convert to list of tuples format
    comp_birth_death_pairs_list = [
        pair
        for pairs_list in comp_birth_death_by_dimension.values()
        for pair in pairs_list
        if pair[1] != np.inf
    ]


    # Calculate distances between each comparison month and the base period
    w_distance = wasserstein_distance(
        diagram_a=comp_birth_death_pairs_list, diagram_b=base_birth_death_pairs_list
    )
    
    distances_dict[month_num] = w_distance

# Plot changes in features (as measured by the Wasserstein distance) over time
plt.figure(figsize=(15, 6))
plt.plot(distances_dict.keys(), distances_dict.values(), marker="s", linestyle="-")
plt.title("Wasserstein Distance Between Months")
plt.xlabel("Month")
plt.ylabel("Distance")
plt.axvline(x=7, color="red", linestyle="--", label="Feature Drift")
plt.grid(True, alpha=0.5)
plt.show()